# University Admission Prediction + QS-Aware Top-K Recommender

## Objetivo

	•	Predecir admitted (0/1) con features académicas + contexto.
	•	Enriquecer con qs_score por (university_clean, mapped_major).
	•	Resolver degree → mapped_major (1 a N) duplicando filas cuando aplique.
	•	Recomendación Top-K usando probabilidad del modelo + QS como señal auxiliar.

### Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from sklearn.preprocessing import LabelEncoder

# Environment

In [2]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR

# Sube hasta encontrar la carpeta que contiene "src"
for _ in range(6):
    if (PROJECT_ROOT / "src").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_EXISTS:", (PROJECT_ROOT / "src").exists())
print("SRC_INIT_EXISTS:", (PROJECT_ROOT / "src" / "__init__.py").exists())

sys.path.insert(0, str(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

NOTEBOOK_DIR: /Users/danimorera/Documents/Portfolio/FLRecomendator/notebooks
PROJECT_ROOT: /Users/danimorera/Documents/Portfolio/FLRecomendator
SRC_EXISTS: True
SRC_INIT_EXISTS: True
sys.path[0]: /Users/danimorera/Documents/Portfolio/FLRecomendator


# Imports and Global Configuration

In [3]:
from src.config import RAW_DIR, RANDOM_STATE
from src.utils import load_excel_safe, load_qs_excel

RAW_DIR, RANDOM_STATE

(PosixPath('/Users/danimorera/Documents/Portfolio/FLRecomendator/data/raw'),
 42)

# Data Sources

In [4]:
STUDENTS_PATH = RAW_DIR / "Five Lands Stats.xlsx"
QS_PATH = RAW_DIR / "University Ranking by Major 2025.xlsx"

STUDENTS_PATH.exists(), QS_PATH.exists()

df_students_raw = load_excel_safe(STUDENTS_PATH)
df_qs_raw = load_qs_excel(QS_PATH)

In [5]:
df_students_raw.columns

Index(['TYPE OF STUDENT', 'LAST YEAR INSTITUTION', 'SCHOOL\nCURRICULUM',
       'SCHOOL\nGRADE', 'GPA', 'TOEFL \n(MAX)', 'SAT\n(SUPERSCORE)',
       'PROFILE\n(4-16)', 'PEAK', 'Unnamed: 9', 'UNIVERSITY', 'COUNTRY',
       'DEGREE', 'AREA OF STUDY', 'Unnamed: 14', 'ADMISSION', 'CURRENCY',
       'TUITION\nYEAR', 'AID\nYEAR', 'AID\nYEAR (EUR)', 'AID\nYEAR (USD)',
       'FINAL TUITION\nYEAR', 'FINAL TUITION\nYEAR (EUR)',
       'FINAL TUITION\nYEAR (USD)', '% AID', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29'],
      dtype='object')

# Tratamiento Students 

In [6]:
from src.cleaning_students import clean_students_schema, null_students_schema

df_students = clean_students_schema(df_students_raw)
df_students = null_students_schema(df_students)


In [7]:
df_students.head()

,school_curriculum,school_grade,gpa,toefl,sat,profile_score,peak,university,country,degree,admitted,sat_required
0,BASE 7,33.0,3.5,102,1000.0,9,0,AMSTERDAM UNIVERSITY OF APPLIED SCIENCE,NETHERLANDS,"Business, Communication, International Busines...",1.0,1
1,BASE 7,33.0,3.5,102,1000.0,9,0,IE UNIVERSITY MADRID CAMPUS,SPAIN,"Business, Communication, International Busines...",1.0,1
2,BASE 7,33.0,3.5,102,1000.0,9,0,UNIVERSIDAD EUROPEA,SPAIN,"Business, Communication, International Busines...",1.0,1
3,BASE 7,33.0,3.5,102,1000.0,9,0,SAINT LOUIS UNIVERSITY MADRID,SPAIN,"Business, Communication, International Busines...",1.0,1
4,BASE 7,33.0,3.5,102,1000.0,9,0,CUNEF UNIVERSIDAD,SPAIN,"Business, Communication, International Busines...",1.0,1


# Tratamiento QS

Lo hago de esta manera debido a que es un ExcelFile y necesito iterar sobre sus sheets

In [8]:
from src.cleaning_qs import initial_clean_qs_schema, augmentation_qs_schema, clean_qs_schema

df_qs = clean_qs_schema(augmentation_qs_schema(initial_clean_qs_schema(df_qs_raw)))

# Mapping df_qs, df_students (degree / universities)

In [9]:
from src.merge_students_qs import university_mapping, degree_mapping, show_degree_mapping

df_students["university"] = [university_mapping(university) for university in df_students["university"]]
df_qs["university"] = [university_mapping(university) for university in df_qs["university"]]

df_students, df_qs_degrees = degree_mapping(df_students)

### Resultado asignacion de variable degree

In [10]:
show_degree_mapping(df_qs_degrees)


Computer Science (15):
  - Computer Science
  - Electronic Engineering with Computer Systems H6G4
  - Business, Cognitive Science, Computer Animation, Computer Science & Web Design, Data Science & Data Analytics, Economics, Engineering, Management
  - Computer Science & Mathematics
  - Mathematics & Computer Science
  - Computer Science and Mathematics with Industrial Experience
  - Computer Engineering
  - Comptuter Science and Artificial Intellience
  - Electrical & Computer Engineering
  - Computer Science and Business
  - Business Information Technology
  - Computing Science & Mathematics
  - Mathematics and Computer Science 3FT BEng (Hon)
  - Mathematics - Computer Science, B.S.
  - Computer Science & Web Design, Engineering, 

Data Science (9):
  - Data & Business Analytics
  - Biosystems Analytics & Technology
  - Ingeniería de Telecomunicaciones  y Business Analytics
  - Business Analytics
  - ADE+ Business Analytics
  - Doble Grado ADE y Ciencias de Datos
  - BBA & Data Analy

### Universidades en comun df_students/df_qs

In [11]:
from src.merge_students_qs import show_inner_universities

show_inner_universities(df_students, df_qs)

Universidades únicas students: 214
Universidades únicas qs: 1758
Universidades en común: 113


In [12]:
from src.merge_students_qs import merge_students_qs

df_pred = merge_students_qs(df_students, df_qs)

In [13]:
df_pred.head()

,school_curriculum,school_grade,gpa,toefl,sat,profile_score,peak,university,country,degree,admitted,sat_required,qs_score
0,BASE 7,33.0,3.5,102,1000.0,9,0,AMSTERDAM UNIVERSITY OF APPLIED SCIENCE,NETHERLANDS,Business,1.0,1,NaN
1,BASE 7,33.0,3.5,102,1000.0,9,0,AMSTERDAM UNIVERSITY OF APPLIED SCIENCE,NETHERLANDS,Politics,1.0,1,NaN
2,BASE 7,33.0,3.5,102,1000.0,9,0,IE UNIVERISTY,SPAIN,Business,1.0,1,NaN
3,BASE 7,33.0,3.5,102,1000.0,9,0,IE UNIVERISTY,SPAIN,Politics,1.0,1,NaN
4,BASE 7,33.0,3.5,102,1000.0,9,0,UNIVERSITY EUROPEA,SPAIN,Business,1.0,1,NaN


In [14]:
df_pred.shape

(751, 13)